# Module 5 Lab — Fine-Grained Authorization for Agents

**Scenario:** a procurement agent proposes purchase orders, while trusted application code resolves identity, relationships, attributes, approvals, and policy immediately before execution.

By the end you will be able to compare a coarse role check with dual user/task authorization, enforce hard and approval-eligible constraints, prevent stale and repeated effects, construct real OpenFGA SDK checks, and interpret labelled safety metrics.

> The model proposes. The PDP decides. The PEP enforces. The effect adapter reports what actually happened.

## 1. Load the tested, credential-free implementation

The notebook imports the same `lab.py` used by the focused tests. It performs no package installation, network request, file write, or live cloud call.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from pathlib import Path
import sys

COURSE_DIR = Path("curriculum/beginner/05-fine-grained-authorization-for-agents").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from lab import (
    CEDAR_POLICY,
    OPENFGA_MODEL,
    REFERENCE_TIME,
    REGO_POLICY,
    DecisionOutcome,
    EffectStatus,
    VendorRecord,
    build_demo_environment,
    demo_identity,
    demo_proposal,
    issue_demo_approval,
    openfga_dual_check_requests,
    run_evaluation,
    unsafe_role_only_authorize,
)

print({name: version(name) for name in ("pydantic", "openfga_sdk", "opa-python-client")})
print("Reference time:", REFERENCE_TIME.isoformat())

## 2. Baseline: a role is too coarse

This intentionally unsafe check knows only a role and action. It ignores the authenticated subject, agent/task binding, tenant, resource, vendor, amount, risk evidence, lifecycle, and call budget.

In [ ]:
unsafe = demo_proposal(
    amount_cents=9_999_999,
    resource_id="department:finance",
    vendor_id="vendor:unknown",
)
assert unsafe_role_only_authorize("ProcurementManager", unsafe) is True
print("Unsafe role-only result:", True)

## 3. Resolve dual authority and current attributes

The trusted identity context is application-owned. The repository separately resolves the user's relationship to the resource, the task's relationship to the action/resource/vendor, and current resource, vendor, and risk records.

In [ ]:
identity, proposal, repository, pep, adapter = build_demo_environment()
decision = repository.preview(identity, proposal, now=REFERENCE_TIME)
print(identity.model_dump(mode="json"))
print(decision.model_dump(mode="json"))
assert decision.outcome is DecisionOutcome.ALLOW
assert len(decision.evidence_versions) == 4

## 4. Every dimension is independently enforced

A valid identity or task does not compensate for an invalid resource, vendor, country, amount, or risk assessment. Reason codes make the boundary observable without exposing hidden reasoning.

In [ ]:
cases = (
    ("valid", {}, {}, DecisionOutcome.ALLOW),
    ("wrong actor", {"actor_id": "agent:other"}, {}, DecisionOutcome.DENY),
    ("wrong resource", {}, {"resource_id": "department:finance"}, DecisionOutcome.DENY),
    ("wrong country", {}, {"country": "GB"}, DecisionOutcome.DENY),
    ("above task limit", {}, {"amount_cents": 1_000_001}, DecisionOutcome.DENY),
)
rows = []
for index, (name, identity_changes, proposal_changes, expected) in enumerate(cases, start=1):
    case_identity = demo_identity().model_copy(update=identity_changes)
    case_proposal = demo_proposal(operation_id=f"OP-MATRIX-{index}").model_copy(update=proposal_changes)
    case_identity, case_proposal, case_repository, _, _ = build_demo_environment(
        identity=case_identity, proposal=case_proposal
    )
    actual = case_repository.preview(case_identity, case_proposal, now=REFERENCE_TIME)
    rows.append((name, expected.value, actual.outcome.value, actual.reason_codes))
    assert actual.outcome is expected
for row in rows:
    print(row)

## 5. Escalation is not authorization

An amount above the autonomous threshold returns `ESCALATE`; the tool is not invoked. A manager approval can satisfy that soft requirement only when the receipt is current, single-use, policy-versioned, and bound to the exact subject, actor, tenant, task, and request digest. Hard prohibitions remain denials.

In [ ]:
high_value = demo_proposal(amount_cents=600_000)
identity, high_value, repository, pep, adapter = build_demo_environment(proposal=high_value)
before = pep.execute(identity, high_value, now=REFERENCE_TIME)
assert before.decision.outcome is DecisionOutcome.ESCALATE
assert before.effect.status is EffectStatus.NOT_ATTEMPTED

approval = issue_demo_approval(identity, high_value, now=REFERENCE_TIME)
after = pep.execute(identity, high_value, approval=approval, now=REFERENCE_TIME)
print(before.decision.reason_codes, after.decision.reason_codes, after.effect.status)
assert after.decision.outcome is DecisionOutcome.ALLOW
assert after.effect.status is EffectStatus.APPLIED

## 6. PEP, idempotency, and actual effects

The PDP decision does not itself prove that a purchase order exists. The PEP calls an observable teaching adapter only after `ALLOW`. An identical retry returns the stored effect; changed arguments under the same operation ID are denied.

In [ ]:
identity, proposal, repository, pep, adapter = build_demo_environment()
first = pep.execute(identity, proposal, now=REFERENCE_TIME)
retry = pep.execute(identity, proposal, now=REFERENCE_TIME)
collision = pep.execute(
    identity, proposal.model_copy(update={"amount_cents": 400_000}), now=REFERENCE_TIME
)
print(first.effect, retry.decision.replayed_decision, collision.decision.reason_codes)
assert adapter.applied_count == 1
assert retry.effect == first.effect
assert collision.decision.outcome is DecisionOutcome.DENY

## 7. Reauthorize at the effect boundary

A preview can become stale. Here the vendor is revoked after preview but before execution. The PEP evaluates the new authoritative version and refuses the effect.

In [ ]:
identity, proposal, repository, pep, adapter = build_demo_environment()
preview = repository.preview(identity, proposal, now=REFERENCE_TIME)
repository.replace_vendor(VendorRecord(
    vendor_id="vendor:acme",
    tenant_id="tenant:oneplusi",
    approved=False,
    sanctioned=False,
    allowed_countries=frozenset({"CA"}),
    version="vendor-v12-revoked",
))
enforced = pep.execute(identity, proposal, now=REFERENCE_TIME)
print(preview.outcome, enforced.decision.outcome, enforced.decision.authorization_epoch)
assert preview.outcome is DecisionOutcome.ALLOW
assert enforced.decision.outcome is DecisionOutcome.DENY
assert adapter.applied_count == 0

## 8. Consume call budgets atomically

Eight distinct operations race for a one-call task grant. Validation and consumption share one authoritative critical section, so exactly one effect is applied. A production deployment needs a transactional shared store rather than an in-process lock.

In [ ]:
proposals = tuple(demo_proposal(operation_id=f"OP-RACE-{index}") for index in range(8))
identity, _, repository, pep, adapter = build_demo_environment(
    proposal=proposals[0], additional_proposals=proposals[1:], max_calls=1
)
with ThreadPoolExecutor(max_workers=8) as pool:
    race = list(pool.map(lambda item: pep.execute(identity, item, now=REFERENCE_TIME), proposals))
allowed = sum(item.decision.outcome is DecisionOutcome.ALLOW for item in race)
denied = sum(item.decision.outcome is DecisionOutcome.DENY for item in race)
print({"population": 8, "allowed": allowed, "denied": denied, "effects": adapter.applied_count})
assert (allowed, denied, adapter.applied_count) == (1, 7, 1)

## 9. Map the invariant to OpenFGA, Cedar, and OPA

The local PDP proves behavior without infrastructure. The next cell constructs real OpenFGA SDK request objects and exposes equivalent Cedar and Rego artifacts. They are integration blueprints, not claims that a live engine ran.

In [ ]:
identity, proposal, _, _, _ = build_demo_environment()
user_check, task_check = openfga_dual_check_requests(identity, proposal)
print({
    "user_check": (user_check.user, user_check.relation, user_check.object),
    "task_check": (task_check.user, task_check.relation, task_check.object),
    "contextual_actor": task_check.contextual_tuples[0].user,
})
print(OPENFGA_MODEL)
print(CEDAR_POLICY)
print(REGO_POLICY)
assert user_check.relation == "user_can_create"
assert task_check.relation == "task_can_create"

## 10. Evaluate with explicit populations

Accuracy alone can hide unsafe access. Report forbidden actions allowed and legitimate work denied with their own denominators; keep escalation cases separate.

In [ ]:
summary = run_evaluation()
print({
    "all_cases": summary.case_count,
    "correct": summary.correct_count,
    "forbidden_population": summary.forbidden_case_count,
    "forbidden_allowed": summary.forbidden_allowed_count,
    "legitimate_population": summary.legitimate_case_count,
    "false_denials": summary.false_denial_count,
    "escalation_population": summary.escalation_case_count,
})
for row in summary.rows:
    print(row)
assert summary.case_count == summary.correct_count == 10
assert summary.forbidden_allowed_count == 0
assert summary.false_denial_count == 0

## 11. Inspect decision evidence

Evidence records trusted identities, exact request/resolved-input digests, policy and source versions, epoch, outcome, and reason codes. A decision event proves policy evaluation—not that an external system applied the effect.

In [ ]:
identity, proposal, repository, pep, _ = build_demo_environment()
result = pep.execute(identity, proposal, now=REFERENCE_TIME)
event = repository.audit_events[0]
print(event.model_dump(mode="json"))
assert event.decision_id == result.decision.decision_id
assert event.policy_version == result.decision.policy_version

## 12. Production upgrades and exercises

Replace the teaching repository with versioned OpenFGA/Cedar/OPA data and a transactional consumption store. Authenticate the PEP-to-PDP channel, define consistency and outage behavior, mask sensitive decision-log fields, reconcile unknown external outcomes, measure authorization latency and revocation propagation, and run policy/schema compatibility tests before rollout.

Exercises:

1. Add an explicit user relationship denial and prove task authority cannot override it.
2. Add session-scoped OpenFGA tuples and compare them with task- and agent-scoped grants.
3. Mutate every Cedar/Rego boundary and prove the test corpus detects widened access.
4. Add a simulated unknown effect outcome and a reconciliation path that never duplicates the purchase order.
5. Design cache keys containing policy version, authorization epoch, request digest, and evidence versions; explain which writes must bypass cache.